# LeetCode #1031: Maximum Sum of Two Non-Overlapping Subarrays

https://leetcode.com/problems/maximum-sum-of-two-non-overlapping-subarrays/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: Prefix Sum + Rolling Max ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Enumerate every pair of non-overlapping subarrays of lengths L and M and track the maximum sum. $O(n^2)$ time after computing prefix sums.

### Optimal: Prefix Sum + Rolling Max ★
Build a prefix sum array. Slide a window of length M (or L) from left to right, keeping a running maximum of the best L-length (or M-length) window seen before the current window. Try both orderings (L before M and M before L) to cover all cases.

**Constraints:**
* $1 \leq firstLen, secondLen \leq 1000$
* $2 \leq nums.length \leq 1000$
* $1 \leq nums[i] \leq 1000$

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxSumTwoNoOverlap(int[] nums, int firstLen, int secondLen) {
        int n = nums.Length;
        int[] prefix = new int[n + 1];
        for (int i = 0; i < n; i++)
            prefix[i + 1] = prefix[i] + nums[i];

        // Helper: max sum of a + b-length windows where a-window ends before b-window starts
        int MaxWith(int a, int b) {
            int result = 0, maxA = 0;
            for (int i = a + b; i <= n; i++) {
                // Best a-window ending at or before position i-b
                maxA = Math.Max(maxA, prefix[i - b] - prefix[i - b - a]);
                // Current b-window sum
                result = Math.Max(result, maxA + prefix[i] - prefix[i - b]);
            }
            return result;
        }

        // Try firstLen before secondLen and secondLen before firstLen
        return Math.Max(MaxWith(firstLen, secondLen), MaxWith(secondLen, firstLen));
    }
}

### Python

In [ ]:
class Solution:
    def max_sum_two_no_overlap(self, nums: list[int], first_len: int, second_len: int) -> int:
        n = len(nums)
        prefix = [0] * (n + 1)
        for i in range(n):
            prefix[i + 1] = prefix[i] + nums[i]

        def max_with(a: int, b: int) -> int:
            # Best a-window sum seen so far before the current b-window
            result = max_a = 0
            for i in range(a + b, n + 1):
                max_a = max(max_a, prefix[i - b] - prefix[i - b - a])
                result = max(result, max_a + prefix[i] - prefix[i - b])
            return result

        # Try both orderings: first_len before second_len and vice versa
        return max(max_with(first_len, second_len), max_with(second_len, first_len))

### Go

In [ ]:
func maxSumTwoNoOverlap(nums []int, firstLen int, secondLen int) int {
    n := len(nums)
    prefix := make([]int, n+1)
    for i, v := range nums {
        prefix[i+1] = prefix[i] + v
    }

    maxWith := func(a, b int) int {
        result, maxA := 0, 0
        for i := a + b; i <= n; i++ {
            // Best a-window ending before the current b-window
            if v := prefix[i-b] - prefix[i-b-a]; v > maxA {
                maxA = v
            }
            if v := maxA + prefix[i] - prefix[i-b]; v > result {
                result = v
            }
        }
        return result
    }

    // Try both orderings: firstLen before secondLen and vice versa
    r1 := maxWith(firstLen, secondLen)
    r2 := maxWith(secondLen, firstLen)
    if r1 > r2 { return r1 }
    return r2
}

### Rust

In [ ]:
impl Solution {
    pub fn max_sum_two_no_overlap(nums: Vec<i32>, first_len: i32, second_len: i32) -> i32 {
        let n = nums.len();
        let mut prefix = vec![0i32; n + 1];
        for i in 0..n {
            prefix[i + 1] = prefix[i] + nums[i];
        }

        let max_with = |a: usize, b: usize| -> i32 {
            let (mut result, mut max_a) = (0, 0);
            for i in (a + b)..=n {
                // Best a-window ending before the current b-window
                max_a = max_a.max(prefix[i - b] - prefix[i - b - a]);
                result = result.max(max_a + prefix[i] - prefix[i - b]);
            }
            result
        };

        // Try both orderings: first_len before second_len and vice versa
        max_with(first_len as usize, second_len as usize)
            .max(max_with(second_len as usize, first_len as usize))
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `nums = [0,6,5,2,2,5,1,9,4]`, `firstLen = 1`, `secondLen = 2`
The best 1-window is at index 7 (value 9) and the best adjacent 2-window is `[1,9]` or `[9,4]`. Trying both orderings, the maximum total is $20$ (windows `[9]` and `[1,9]`... actually `[5,1]`+`[9,4]` = 19 vs `[9]`+`[5,1]`= 15; best is `[9]`+`[5,2]`... answer is $20$).

### 2. Slightly Complex
**Input:** `nums = [3,8,1,3,2,1,8,9,0]`, `firstLen = 3`, `secondLen = 2`
The rolling max tracks the best 3-window before the current 2-window and vice versa. The maximum is $3+8+1 + 8+9 = 29$.

### 3. Edge Case: Time Factor
**Input:** `nums = [1,1,...,1]` ($n = 1000$), `firstLen = 500`, `secondLen = 500`
All elements are 1; every window sum is equal. The loop runs $n - (L+M) + 1 = 1$ time — fastest possible inner loop, $O(n)$ overall.

### 4. Edge Case: Space Factor
**Input:** `nums` has 1000 elements, all values 1000
The prefix array is the only extra space ($O(n)$). With all windows equal the rolling max hits its maximum value immediately and stays there.

### 5. Almost-Impossible but Plausible
**Input:** `nums = [1000,1000,...,1000]`, `firstLen = 499`, `secondLen = 500`
Two adjacent windows cover 999 of the 1000 positions. The algorithm finds the pair starting at index 0 and index 499, giving $499 \times 1000 + 500 \times 1000 = 999{,}000$.